In [2]:
import time
import pandas as pd
from transformers import pipeline

DEBERTA_MODEL_NAME = (
    "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
)

deberta_classifier = pipeline(
    task="zero-shot-classification",
    model=DEBERTA_MODEL_NAME
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [3]:
deberta_predictions = []
deberta_confidences = []
deberta_latencies = []

In [4]:
evaluation_df = pd.read_csv(
    "data/job_posts_evaluation.csv"
)

evaluation_df

,job_description,expected_category
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines
2,Build machine learning models for text classif...,artificial intelligence and machine learning
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation
5,Create automated release pipelines using Docke...,devops and deployment automation
6,"Design scalable AWS infrastructure using EC2, ...",cloud infrastructure and cloud engineering
7,Build secure Azure cloud environments and mana...,cloud infrastructure and cloud engineering
8,Monitor security events using SIEM tools and i...,cybersecurity and information security
9,"Perform vulnerability assessments, threat dete...",cybersecurity and information security


In [5]:
deberta_predictions = []
deberta_confidences = []
deberta_latencies = []

In [6]:
candidate_labels = [
    "data engineering and data pipelines",
    "artificial intelligence and machine learning",
    "devops and deployment automation",
    "cloud infrastructure and cloud engineering",
    "cybersecurity and information security",
    "software and application development",
]

In [7]:
for job_description in evaluation_df["job_description"]:
    start_time = time.perf_counter()

    result = deberta_classifier(
        job_description,
        candidate_labels=candidate_labels,
        multi_label=False
    )

    latency = time.perf_counter() - start_time

    deberta_predictions.append(result["labels"][0])
    deberta_confidences.append(result["scores"][0] * 100)
    deberta_latencies.append(latency)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [8]:
deberta_results_df = evaluation_df.copy()

deberta_results_df["predicted_category"] = deberta_predictions
deberta_results_df["confidence_percent"] = deberta_confidences
deberta_results_df["latency_seconds"] = deberta_latencies

deberta_results_df["is_correct"] = (
    deberta_results_df["expected_category"]
    == deberta_results_df["predicted_category"]
)

In [9]:
deberta_results_df[
    [
        "job_description",
        "expected_category",
        "predicted_category",
        "confidence_percent",
        "latency_seconds",
        "is_correct",
    ]
]

,job_description,expected_category,predicted_category,confidence_percent,latency_seconds,is_correct
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines,data engineering and data pipelines,75.765580,1.184707,True
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines,data engineering and data pipelines,98.357749,0.405419,True
2,Build machine learning models for text classif...,artificial intelligence and machine learning,artificial intelligence and machine learning,59.862334,0.121493,True
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning,software and application development,43.493381,0.114121,False
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation,devops and deployment automation,72.826284,0.117846,True
5,Create automated release pipelines using Docke...,devops and deployment automation,devops and deployment automation,72.312367,0.111469,True
6,"Design scalable AWS infrastructure using EC2, ...",cloud infrastructure and cloud engineering,cloud infrastructure and cloud engineering,88.342428,0.112116,True
7,Build secure Azure cloud environments and mana...,cloud infrastructure and cloud engineering,cloud infrastructure and cloud engineering,54.542512,0.107063,True
8,Monitor security events using SIEM tools and i...,cybersecurity and information security,cybersecurity and information security,96.529382,0.106636,True
9,"Perform vulnerability assessments, threat dete...",cybersecurity and information security,cybersecurity and information security,88.991016,0.111498,True


In [10]:
deberta_accuracy = (
    deberta_results_df["is_correct"].mean() * 100
)

deberta_average_latency = (
    deberta_results_df["latency_seconds"].mean()
)

deberta_average_confidence = (
    deberta_results_df["confidence_percent"].mean()
)

print(f"Accuracy: {deberta_accuracy:.2f}%")
print(f"Average latency: {deberta_average_latency:.3f} seconds")
print(f"Average confidence: {deberta_average_confidence:.2f}%")

Accuracy: 91.67%
Average latency: 0.227 seconds
Average confidence: 78.77%


In [11]:
import pandas as pd

MODEL_NAME = "DeBERTa"
MODEL_ID = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

COST_USD = 0.0
COST_DESCRIPTION = "Free - Local inference"

deberta_final_result = pd.DataFrame([
    {
        "model_name": MODEL_NAME,
        "model_id": MODEL_ID,
        "accuracy_percent": round(
            float(deberta_accuracy),
            2
        ),
        "average_latency_seconds": round(
            float(deberta_average_latency),
            4
        ),
        "average_confidence_percent": round(
            float(deberta_average_confidence),
            2
        ),
        "cost_usd": COST_USD,
        "cost_description": COST_DESCRIPTION,
        "correct_predictions": int(
            deberta_results_df["is_correct"].sum()
        ),
        "wrong_predictions": int(
            (~deberta_results_df["is_correct"]).sum()
        ),
        "total_examples": int(
            len(deberta_results_df)
        ),
    }
])

deberta_final_result

,model_name,model_id,accuracy_percent,average_latency_seconds,average_confidence_percent,cost_usd,cost_description,correct_predictions,wrong_predictions,total_examples
0,DeBERTa,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,91.67,0.2268,78.77,0.0,Free - Local inference,11,1,12


In [17]:
from pathlib import Path
import pandas as pd

RESULTS_FILE = Path("data/final_result_of_comparison.csv")

comparison_df = pd.read_csv(RESULTS_FILE)

# توحيد accuracy القديمة وتحويلها من 0.8333 إلى 83.33
if "accuracy" in comparison_df.columns:
    old_accuracy_percent = comparison_df["accuracy"] * 100

    if "accuracy_percent" not in comparison_df.columns:
        comparison_df["accuracy_percent"] = old_accuracy_percent
    else:
        comparison_df["accuracy_percent"] = (
            comparison_df["accuracy_percent"]
            .fillna(old_accuracy_percent)
        )

    comparison_df = comparison_df.drop(columns=["accuracy"])

# إضافة عمود Average Confidence لو مش موجود
if "average_confidence_percent" not in comparison_df.columns:
    comparison_df["average_confidence_percent"] = pd.NA

# ترتيب الأعمدة
FINAL_COLUMNS = [
    "model_name",
    "model_id",
    "accuracy_percent",
    "average_latency_seconds",
    "average_confidence_percent",
    "cost_usd",
    "cost_description",
    "correct_predictions",
    "wrong_predictions",
    "total_examples",
]

comparison_df = comparison_df[FINAL_COLUMNS]

comparison_df = comparison_df.sort_values(
    by="accuracy_percent",
    ascending=False
).reset_index(drop=True)

comparison_df.to_csv(
    RESULTS_FILE,
    index=False
)

print("Comparison file columns fixed successfully.")

comparison_df

Comparison file columns fixed successfully.


,model_name,model_id,accuracy_percent,average_latency_seconds,average_confidence_percent,cost_usd,cost_description,correct_predictions,wrong_predictions,total_examples
0,DeBERTa,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,91.67,0.2268,78.77,0.0,Free - Local inference,11,1,12
1,ModernBERT,tasksource/ModernBERT-base-nli,83.33,0.0977,41.82,0.0,Free - Local inference,10,2,12
